In [8]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

# === Config: STC Bank (Google Play) ===
app_id = "sa.com.stcbank"   # package id from the URL
language = "en"             # e.g., "en" or "ar"
country = "sa"              # Saudi Arabia store
sort_order = Sort.MOST_RELEVANT

# Pagination / limits
review_count = 50           # reviews per request (max ≈ 200)
desired_total_reviews = 1500  # stop after this many (or when no more pages)

# Star filter: choose 1..5 or set to None for all
filter_stars = 5

# Output
excel_file = f"stc_bank_{language}_{country}_{filter_stars}star_reviews.xlsx"

all_reviews = []

# ----- Initial fetch -----
result, continuation_token = reviews(
    app_id,
    lang=language,
    country=country,
    sort=sort_order,
    count=review_count,
    filter_score_with=filter_stars  # restrict to selected star rating
)
all_reviews.extend(result)
print(f"Fetched {len(all_reviews)} reviews so far...")

# ----- Continue fetching until limit or no more pages -----
while continuation_token and len(all_reviews) < desired_total_reviews:
    more_reviews, continuation_token = reviews(
        app_id,
        lang=language,
        country=country,
        sort=sort_order,
        count=min(review_count, desired_total_reviews - len(all_reviews)),
        continuation_token=continuation_token,
        filter_score_with=filter_stars
    )
    all_reviews.extend(more_reviews)
    print(f"Fetched {len(all_reviews)} reviews so far...")
    time.sleep(1)  # be polite

# ----- Build DataFrame -----
df = pd.DataFrame(all_reviews)

# Keep common fields; add safe copies to avoid SettingWithCopy warnings
cols = ["content", "score", "thumbsUpCount", "at", "reviewId", "userName", "reviewCreatedVersion"]
available = [c for c in cols if c in df.columns]
df_out = df[available].copy()

df_out = df_out.rename(columns={
    "content": "Review_Text",
    "score": "Rating",
    "thumbsUpCount": "Helpful_Votes",
    "at": "Review_Time_UTC",
    "reviewCreatedVersion": "App_Version"
})

# Make timestamps Excel-friendly (ISO 8601 strings)
if "Review_Time_UTC" in df_out.columns:
    df_out["Review_Time_UTC"] = pd.to_datetime(df_out["Review_Time_UTC"], errors="coerce", utc=True)\
                                   .dt.strftime("%Y-%m-%d %H:%M:%S")

# ----- Export to Excel -----
# Requires: pip install openpyxl
df_out.to_excel(excel_file, index=False)
print(f"✅ Exported {len(df_out)} {filter_stars}-star reviews to {excel_file}")

Fetched 50 reviews so far...
Fetched 100 reviews so far...
Fetched 150 reviews so far...
Fetched 200 reviews so far...
Fetched 250 reviews so far...
Fetched 300 reviews so far...
Fetched 350 reviews so far...
Fetched 400 reviews so far...
Fetched 450 reviews so far...
Fetched 500 reviews so far...
Fetched 550 reviews so far...
Fetched 600 reviews so far...
Fetched 650 reviews so far...
Fetched 700 reviews so far...
Fetched 750 reviews so far...
Fetched 800 reviews so far...
Fetched 850 reviews so far...
Fetched 900 reviews so far...
Fetched 950 reviews so far...
Fetched 1000 reviews so far...
Fetched 1050 reviews so far...
Fetched 1100 reviews so far...
Fetched 1150 reviews so far...
Fetched 1200 reviews so far...
Fetched 1250 reviews so far...
Fetched 1300 reviews so far...
Fetched 1350 reviews so far...
Fetched 1400 reviews so far...
Fetched 1450 reviews so far...
Fetched 1500 reviews so far...
✅ Exported 1500 5-star reviews to stc_bank_en_sa_5star_reviews.xlsx
